# LoRaS-CT — With KD vs Without KD (LC2500 Clean + NCT7k Disjoint)

Standalone notebook. Scope: **LoRaS-CT only**, trained two ways per dataset —
**With KD** (distilled from ImageNet-pretrained ViT/DeiT/Swin teachers, logits
simply averaged, teachers NOT fine-tuned on the target dataset) and
**Without KD** (plain cross-entropy training on ground-truth labels only, no
teachers) — across **two datasets**:

- **LC2500 (Clean)** — LC25000 (lung + colon histopathology). The on-disk
  "Train and Validation Set" / "Test Set" folders are **not** an official
  split from the dataset authors -- they were split at the image level, which
  leaks near-duplicate augmentations of the same original tissue crop across
  train/test. We pool both folders and re-split at the **original-image
  ("prototype") level** using the group mapping from
  [GeorgeBatch/LC25000-clean](https://github.com/GeorgeBatch/LC25000-clean),
  so no augmented sibling of a given original tile crosses a split boundary.
- **NCT7k (Disjoint)** — train/val on **NCT-CRC-HE-100K**, test on
  **CRC-VAL-HE-7K**. These come from genuinely separate patient slides (no
  augmentation-duplicate mechanism), so the official split is kept as-is.

Single seed (42), no multi-seed statistics. Both LoRaS-CT variants are
trained and checkpointed independently per dataset.

## 1. Setup

In [ ]:
!pip install --upgrade pip setuptools wheel --quiet
!pip install numpy pandas matplotlib torch torchvision timm scikit-learn thop tqdm --quiet


!pip uninstall -y torch torchvision torchaudio

!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import torch
import torch.multiprocessing as mp
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass  # already set (e.g. re-running this cell) -- harmless
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split, Subset
from timm import create_model
from timm.layers import DropPath
from thop import profile
from tqdm.auto import tqdm

# ============================================================
# GLOBAL CONFIG
# ============================================================
GLOBAL_NUM_LAYERS = 2
GLOBAL_NUM_HEADS = 8
GLOBAL_RANK = 32
GLOBAL_NUM_EPOCHS = 10  # LoRaS-CT student training epochs (both KD and No-KD)

assert GLOBAL_RANK % GLOBAL_NUM_HEADS == 0, "GLOBAL_RANK must be divisible by GLOBAL_NUM_HEADS"

# Set True to ignore any saved checkpoints and force everything to retrain from scratch.
FORCE_RETRAIN = True

# Single seed -- no multi-seed statistics needed for this comparison.
SEED = 42

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)

# ============================================================
# QUICK TEST MODE -- smoke-test the whole pipeline in a few minutes before
# committing to the full run. Tiny data subsets + fewer epochs.
# ============================================================
QUICK_TEST_MODE = False
QUICK_TEST_MAX_TRAIN = 64
QUICK_TEST_MAX_VAL = 16
QUICK_TEST_MAX_TEST = 16
RUN_EPOCHS = 2 if QUICK_TEST_MODE else GLOBAL_NUM_EPOCHS

def quick_subset(torch_dataset, max_n):
    if not QUICK_TEST_MODE:
        return torch_dataset
    n = min(max_n, len(torch_dataset))
    return Subset(torch_dataset, list(range(n)))

if QUICK_TEST_MODE:
    print("QUICK_TEST_MODE is ON -- using tiny data subsets / fewer epochs to smoke-test the pipeline.")


## 2. Dataset registry

In [ ]:
# ============================================================
# DATASET REGISTRY
#
#   LC2500_Clean : LC25000 -- pooled + re-split at the group (original-tile)
#                  level via the LC25000-clean mapping (see the group-mapping
#                  fetch cell right below). "path" fields below are read but
#                  then POOLED TOGETHER and re-split -- the on-disk
#                  Train/Test boundary is discarded (see Section 5b).
#   NCT7k_Disjoint : official NCT-CRC-HE-100K (train/val) vs CRC-VAL-HE-7K
#                    (test) split -- genuinely disjoint patient slides, kept
#                    as-is.
# ============================================================
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
DATA_ROOT = "/kaggle/input" if IS_KAGGLE else os.environ.get("DATA_ROOT", "./data")
OUTPUT_ROOT = "/kaggle/working" if IS_KAGGLE else os.environ.get("OUTPUT_ROOT", "./outputs")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

DATASET_CONFIGS = {
    "LC2500_Clean": {
        "train_val_path": "/scratch/home/admin/lung_colon_image_set/Train and Validation Set",   # <-- edit if needed
        "test_path": "/scratch/home/admin/lung_colon_image_set/Test Set",                          # <-- edit if needed
        "lc25000_grouped": True,   # triggers the group-aware pool+re-split in Section 8 below
    },
    "NCT7k_Disjoint": {
        "train_val_path": "/scratch/home/admin/NCT-CRC-HE-100K/",   # <-- edit if needed
        "test_path": "/scratch/home/admin/CRC-VAL-HE-7K",           # <-- edit if needed
    },
}

def _check_dataset_paths(cfg):
    paths = [cfg["path"]] if "path" in cfg else [cfg["train_val_path"], cfg["test_path"]]
    return [p for p in paths if not os.path.isdir(p)]

# ============================================================
# DATASET SELECTOR
#   - one name from DATASET_CONFIGS  -> runs the pipeline for just that dataset
#   - "ALL"                          -> runs every dataset, back-to-back
# ============================================================
SELECTED_DATASET = "ALL"   # <-- "LC2500_Clean" | "NCT7k_Disjoint" | "ALL"

assert SELECTED_DATASET == "ALL" or SELECTED_DATASET in DATASET_CONFIGS, (
    f"Unknown SELECTED_DATASET '{SELECTED_DATASET}'. Choose one of {list(DATASET_CONFIGS)} or 'ALL'."
)
DATASETS_TO_RUN = list(DATASET_CONFIGS.keys()) if SELECTED_DATASET == "ALL" else [SELECTED_DATASET]
print(f"Will run the pipeline for: {DATASETS_TO_RUN}")


## 3. Group mapping for leakage-safe LC25000 split

In [ ]:
# ============================================================
# Fetch LC25000-clean group mapping (fixes LC25000 augmentation-duplicate
# leakage). Source: https://github.com/GeorgeBatch/LC25000-clean
#
# We only need ONE file from that repo: kaggle/lc25000_image_groups.csv
# (~1.2 MB) -- no need to clone the whole thing.
# ============================================================
import os as _os
import urllib.request as _urlreq

# --- If you already have the CSV somewhere (e.g. via Kaggle "+ Add Data" ->
# --- search "LC25000 Clean Image Groups" / gbatchkala/lc25000-clean-groups,
# --- OR you downloaded it elsewhere and scp'd it over), point this at it.
# --- Leave as None to auto-download from GitHub instead (requires this
# --- notebook's internet to be ON -- it already must be, since the pip
# --- installs in cell 1 need it too).
GROUPS_CSV_OVERRIDE = None   # e.g. "/kaggle/input/lc25000-clean-groups/lc25000_image_groups.csv"

GROUPS_CSV_URL = "https://raw.githubusercontent.com/GeorgeBatch/LC25000-clean/main/kaggle/lc25000_image_groups.csv"
# NOTE: never point the download target at /kaggle/input/... -- that mount is
# READ-ONLY. Downloads always land in a writable spot (OUTPUT_ROOT).
GROUPS_CSV_DOWNLOAD_TARGET = _os.path.join(OUTPUT_ROOT, "lc25000_image_groups.csv")

_need_groups_csv = any(DATASET_CONFIGS[d].get("lc25000_grouped") for d in DATASETS_TO_RUN)

if GROUPS_CSV_OVERRIDE and _os.path.isfile(GROUPS_CSV_OVERRIDE):
    GROUPS_CSV = GROUPS_CSV_OVERRIDE
elif _os.path.isfile(GROUPS_CSV_DOWNLOAD_TARGET):
    GROUPS_CSV = GROUPS_CSV_DOWNLOAD_TARGET
else:
    GROUPS_CSV = GROUPS_CSV_DOWNLOAD_TARGET
    if _need_groups_csv:
        print(f"Downloading group mapping to {GROUPS_CSV} ...")
        try:
            _urlreq.urlretrieve(GROUPS_CSV_URL, GROUPS_CSV)
            print("Download succeeded.")
        except Exception as e:
            print(f"Auto-download failed ({e}).")
            print(
                "Two likely causes:\n"
                "  A) This environment's internet is OFF / restricted. On Kaggle: "
                "notebook side panel -> Settings -> Internet -> toggle ON, then re-run this cell.\n"
                "  B) No internet at all in this environment. Manual fallback:\n"
                "       1. On any machine with internet:\n"
                f"            curl -L -o lc25000_image_groups.csv {GROUPS_CSV_URL}\n"
                "       2. scp/copy that file to this machine and set:\n"
                "            GROUPS_CSV_OVERRIDE = \"/path/to/lc25000_image_groups.csv\"\n"
                "          above and re-run this cell."
            )

if _need_groups_csv:
    assert _os.path.isfile(GROUPS_CSV), (
        f"Group mapping still not found at {GROUPS_CSV}. See instructions printed "
        f"above (or re-run after fixing internet access / manual copy)."
    )
    print(f"Group mapping ready at {GROUPS_CSV}")


### 3a. `PathListDataset` -- lets us discard the on-disk LC25000 split and build our own

In [ ]:
%%writefile lc25000_dataset_utils.py
# Written to disk (not defined inline) because DataLoader workers use
# multiprocessing 'spawn' (set earlier in this notebook) -- spawned worker
# processes need to re-IMPORT the Dataset class from an actual module; a
# class defined in a notebook cell lives in __main__ and can't be
# pickled/found by them, causing
# "Can't get attribute 'PathListDataset' on <module '__main__'>".
import torch
from PIL import Image


class PathListDataset(torch.utils.data.Dataset):
    """Dataset built from an explicit (filepath, label_idx) list -- lets us
    ignore the pre-existing folder split entirely and build our own,
    leakage-safe one."""
    def __init__(self, samples, transform=None):
        self.samples = samples  # list of (path, label_idx)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, label


class ClassesShim:
    """Minimal stand-in for ImageFolder's .classes attribute, since other
    cells reference train_val_dataset.classes / num_classes."""
    def __init__(self, classes):
        self.classes = classes


In [ ]:
from lc25000_dataset_utils import PathListDataset, ClassesShim


## 4. Transforms

In [ ]:
train_val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

batch_size = QUICK_TEST_MAX_TRAIN if QUICK_TEST_MODE else 32


## 5. Model definitions (LoRaS-CT only)

In [ ]:
class ResNet18_Features(nn.Module):
    def __init__(self):
        super(ResNet18_Features, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-2])

    def forward(self, x):
        return self.features(x)  # (B, 512, 7, 7)


class DenseNet121_Features(nn.Module):
    def __init__(self):
        super(DenseNet121_Features, self).__init__()
        densenet = models.densenet121(pretrained=True)
        self.features = densenet.features

    def forward(self, x):
        x = self.features(x)
        x = F.relu(x, inplace=False)
        return x  # (B, 1024, 7, 7)


In [ ]:
class LowRankSparseMultiheadAttention(nn.Module):
    """Attention computed entirely in rank-space (dimension r) -- Q, K, V are
    never projected back up to full embed_dim before the attention product."""
    def __init__(self, embed_dim, num_heads, rank, sparsity_ratio=0.5):
        super(LowRankSparseMultiheadAttention, self).__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        assert rank % num_heads == 0, "rank must be divisible by num_heads (rank-space multi-head split)"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.rank = rank
        self.rank_head_dim = rank // num_heads
        self.sparsity_ratio = sparsity_ratio

        self.q_low = nn.Linear(embed_dim, rank, bias=False)
        self.k_low = nn.Linear(embed_dim, rank, bias=False)
        self.v_low = nn.Linear(embed_dim, rank, bias=False)
        self.rank_mix = nn.Linear(rank, rank, bias=False)
        self.out_proj = nn.Linear(rank, embed_dim, bias=False)
        self.scale = rank ** -0.5

    def sparse_attention(self, attn_scores, sparsity_ratio):
        """Masks non-top-k positions with -inf BEFORE softmax -- multiplying by 0
        and softmaxing would still give masked positions exp(0)=1, which can
        outweigh genuinely-kept positions when raw scores are negative."""
        batch_size, num_heads, seq_length, _ = attn_scores.size()
        if seq_length == 1:
            return attn_scores
        num_to_keep = max(1, int(sparsity_ratio * seq_length))
        top_scores, _ = torch.topk(attn_scores, k=num_to_keep, dim=-1)
        threshold = top_scores.min(dim=-1, keepdim=True)[0]
        sparse_mask = attn_scores >= threshold
        return attn_scores.masked_fill(~sparse_mask, float('-inf'))

    def forward(self, x):
        batch_size, seq_length, embed_dim = x.size()
        Q = self.q_low(x)
        K = self.k_low(x)
        V = self.v_low(x)
        Q = Q.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        sparse_attn_scores = self.sparse_attention(attn_scores, self.sparsity_ratio)
        attn_probs = F.softmax(sparse_attn_scores, dim=-1)
        attn_output = torch.matmul(attn_probs, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.rank)
        attn_output = self.rank_mix(attn_output)
        return self.out_proj(attn_output)


class CustomDeiTLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, rank, mlp_ratio=4., drop_path=0.1, sparsity_ratio=0.5):
        super(CustomDeiTLayer, self).__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = LowRankSparseMultiheadAttention(embed_dim, num_heads, rank, sparsity_ratio)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim), nn.GELU(), nn.Linear(mlp_hidden_dim, embed_dim),
        )

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class HybridStudentModel(nn.Module):
    """LoRaS-CT."""
    def __init__(self, num_classes, embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS,
                 rank=GLOBAL_RANK, drop_path_rate=0.1, sparsity_ratio=0.5, grid_size=3):
        super(HybridStudentModel, self).__init__()
        self.resnet = ResNet18_Features()
        self.densenet = DenseNet121_Features()
        self.grid_size = grid_size
        concat_channels = 512 + 1024

        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer(embed_dim, num_heads, rank, drop_path=drop_path_rate,
                             sparsity_ratio=sparsity_ratio) for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(concat_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        resnet_feats = self.resnet(x)
        densenet_feats = self.densenet(x)
        combined_feats = torch.cat((resnet_feats, densenet_feats), dim=1)
        if self.grid_size != combined_feats.shape[-1]:
            combined_feats = F.adaptive_avg_pool2d(combined_feats, (self.grid_size, self.grid_size))
        b, c, h, w = combined_feats.shape
        combined_feats = combined_feats.view(b, c, h * w).permute(0, 2, 1)
        x = self.deit_embed(combined_feats)
        for layer in self.deit_layers:
            x = layer(x)
        x = self.norm(x)
        pooled = x.mean(dim=1)
        return self.classifier(pooled)


## 6. Shared utilities (train/eval)

In [ ]:
def evaluate_model(model, data_loader, criterion, return_predictions=False):
    model.eval()
    correct, total, test_loss = 0, 0, 0.0
    all_labels, all_preds = [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.cuda(), labels.cuda()
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            if return_predictions:
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(predicted.cpu().numpy())
    accuracy = 100 * correct / total
    avg_loss = test_loss / len(data_loader)
    if return_predictions:
        return accuracy, avg_loss, np.array(all_labels), np.array(all_preds)
    return accuracy, avg_loss


class DistillationLoss(nn.Module):
    def __init__(self, alpha=0.5, temperature=3.0):
        super(DistillationLoss, self).__init__()
        self.alpha = alpha
        self.temperature = temperature
        self.ce_loss = nn.CrossEntropyLoss()
        self.kl_div = nn.KLDivLoss(reduction="batchmean")

    def forward(self, student_logits, teacher_logits, ground_truth):
        hard_loss = self.ce_loss(student_logits, ground_truth)
        soft_loss = self.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=1),
            F.softmax(teacher_logits / self.temperature, dim=1)
        ) * (self.temperature ** 2)
        return self.alpha * soft_loss + (1 - self.alpha) * hard_loss


def train_model_with_distillation(student_model, teacher_models, train_loader, val_loader,
                                   distillation_criterion, optimizer, num_epochs=1, verbose=False):
    for epoch in range(num_epochs):
        student_model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"  epoch {epoch+1}/{num_epochs}", leave=False)
        for images, labels in pbar:
            images, labels = images.cuda(), labels.cuda()
            optimizer.zero_grad()
            student_outputs = student_model(images)
            with torch.no_grad():
                teacher_logits = [teacher(images) for teacher in teacher_models]
                combined_teacher_logits = sum(teacher_logits) / len(teacher_logits)
            loss = distillation_criterion(student_outputs, combined_teacher_logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        avg_loss = running_loss / len(train_loader)
        if verbose:
            val_accuracy, val_loss = evaluate_model(student_model, val_loader, distillation_criterion.ce_loss)
            print(f"  epoch [{epoch+1}/{num_epochs}] train_loss={avg_loss:.4f} "
                  f"val_loss={val_loss:.4f} val_acc={val_accuracy:.2f}%")
        else:
            print(f"  epoch [{epoch+1}/{num_epochs}] train_loss={avg_loss:.4f}")
    return student_model


def train_model_plain(model, train_loader, val_loader, criterion, optimizer, num_epochs=1, verbose=False):
    """Plain (non-distillation) training -- used for teacher fine-tuning AND
    for the 'Without KD' LoRaS-CT variant (ground-truth labels only)."""
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"  epoch {epoch+1}/{num_epochs}", leave=False)
        for images, labels in pbar:
            images, labels = images.cuda(), labels.cuda()
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        avg_loss = running_loss / len(train_loader)
        if verbose:
            val_accuracy, val_loss = evaluate_model(model, val_loader, criterion)
            print(f"  epoch [{epoch+1}/{num_epochs}] train_loss={avg_loss:.4f} "
                  f"val_loss={val_loss:.4f} val_acc={val_accuracy:.2f}%")
        else:
            print(f"  epoch [{epoch+1}/{num_epochs}] train_loss={avg_loss:.4f}")
    return model


## 7. Checkpointing

In [ ]:
# ============================================================
# Per-model checkpointing -- keyed by (dataset, model_name), covers teachers
# AND both LoRaS-CT variants. An interruption anywhere doesn't require
# retraining what already finished.
# ============================================================
def checkpoint_path(dataset_name, model_name):
    safe_name = model_name.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "-")
    return f"{OUTPUT_ROOT}/{dataset_name}_{safe_name}_checkpoint.pth"


def save_checkpoint(dataset_name, model_name, model, extra=None):
    path = checkpoint_path(dataset_name, model_name)
    payload = {'model_state_dict': model.state_dict()}
    if extra is not None:
        payload['extra'] = extra
    torch.save(payload, path)
    print(f"  [checkpoint saved] {path}")


def load_checkpoint(dataset_name, model_name, model):
    """Loads weights into `model` in-place if a checkpoint exists. Returns the
    'extra' payload (or an empty dict) on success, None if no checkpoint was
    loaded (caller should train). Respects FORCE_RETRAIN."""
    path = checkpoint_path(dataset_name, model_name)
    if FORCE_RETRAIN or not os.path.exists(path):
        return None
    ckpt = torch.load(path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"  [checkpoint found] Skipping training for '{model_name}' -- loaded from {path}")
    return ckpt.get('extra', {})


## 8. Per-dataset pipeline

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from collections import Counter

def load_dataset(dataset_name, dataset_cfg, seed):
    """Returns train_loader, val_loader, test_loader, num_classes."""
    g = torch.Generator().manual_seed(seed)
    train_val_path = dataset_cfg["train_val_path"]
    test_path = dataset_cfg["test_path"]

    if dataset_cfg.get("lc25000_grouped"):
        # ------------------------------------------------------------------
        # LC25000 was built by augmenting ~250 original slide crops per class
        # up to 5,000 images each (~20 near-duplicate rotations/flips per
        # original -- see Borkowski et al. 2019, arXiv:1912.12142). There is
        # NO official train/test split from the dataset authors: the
        # "Train and Validation Set" / "Test Set" folders on disk were split
        # at the IMAGE level, which puts near-duplicate augmentations of the
        # same original tissue crop on both sides of the split -> leakage ->
        # inflated, near-100% accuracy for every model.
        #
        # Fix: pool ALL images from both folders, discard that split
        # entirely, and re-split at the ORIGINAL-IMAGE ("prototype") level
        # using the group mapping from LC25000-clean
        # (https://github.com/GeorgeBatch/LC25000-clean), so every augmented
        # sibling of a given original stays on the same side of the split.
        # ------------------------------------------------------------------
        groups_df = pd.read_csv(GROUPS_CSV)
        fname_to_group = dict(zip(groups_df["filename"], groups_df["group_id"]))

        class_names = sorted(d.name for d in os.scandir(train_val_path) if d.is_dir())
        test_class_names = sorted(d.name for d in os.scandir(test_path) if d.is_dir())
        assert class_names == test_class_names, (
            f"[{dataset_name}] Class mismatch between train/val and test folders: "
            f"{class_names} vs {test_class_names}."
        )
        class_to_idx = {c: i for i, c in enumerate(class_names)}
        num_classes = len(class_names)
        print(f"[{dataset_name}] Classes ({num_classes}): {class_names}")

        all_paths, all_groups, all_labels = [], [], []
        missing_from_map = 0

        for root in (train_val_path, test_path):
            for cname in class_names:
                class_dir = os.path.join(root, cname)
                if not os.path.isdir(class_dir):
                    continue
                for fname in os.listdir(class_dir):
                    if not fname.lower().endswith((".jpeg", ".jpg", ".png")):
                        continue
                    gid = fname_to_group.get(fname)
                    if gid is None:
                        missing_from_map += 1
                        continue  # can't safely place this image without a group id
                    all_paths.append(os.path.join(class_dir, fname))
                    all_groups.append(gid)
                    all_labels.append(class_to_idx[cname])

        print(f"[{dataset_name}] Pooled {len(all_paths)} images "
              f"({missing_from_map} skipped -- filename not found in group mapping; "
              f"investigate before proceeding if this is more than a handful).")

        all_groups = np.array(all_groups)
        all_labels = np.array(all_labels)
        all_paths = np.array(all_paths, dtype=object)

        # Step 1: hold out ~1/6 (~16.7%) of GROUPS (not images) for test, stratified by class.
        N_TEST_SPLITS = 6
        sgkf_test = StratifiedGroupKFold(n_splits=N_TEST_SPLITS, shuffle=True, random_state=seed)
        trainval_idx, test_idx = next(sgkf_test.split(all_paths, all_labels, groups=all_groups))

        # Step 2: split remaining groups into train / val (~90/10 of trainval).
        N_VAL_SPLITS = 10
        sgkf_val = StratifiedGroupKFold(n_splits=N_VAL_SPLITS, shuffle=True, random_state=seed)
        train_sub_idx, val_sub_idx = next(sgkf_val.split(
            all_paths[trainval_idx], all_labels[trainval_idx], groups=all_groups[trainval_idx]
        ))
        train_idx = trainval_idx[train_sub_idx]
        val_idx = trainval_idx[val_sub_idx]

        # Sanity check: zero group overlap across splits (this is the whole point).
        g_train, g_val, g_test = set(all_groups[train_idx]), set(all_groups[val_idx]), set(all_groups[test_idx])
        assert not (g_train & g_val), "Group leakage between train and val!"
        assert not (g_train & g_test), "Group leakage between train and test!"
        assert not (g_val & g_test), "Group leakage between val and test!"

        print(f"[{dataset_name}] Train: {len(train_idx):>6} imgs / {len(g_train):>4} groups")
        print(f"[{dataset_name}] Val:   {len(val_idx):>6} imgs / {len(g_val):>4} groups")
        print(f"[{dataset_name}] Test:  {len(test_idx):>6} imgs / {len(g_test):>4} groups")
        print(f"[{dataset_name}] Group overlap across splits: NONE (verified).")
        print(f"[{dataset_name}] Class balance (train):", Counter(all_labels[train_idx]))
        print(f"[{dataset_name}] Class balance (val):  ", Counter(all_labels[val_idx]))
        print(f"[{dataset_name}] Class balance (test): ", Counter(all_labels[test_idx]))

        train_dataset = PathListDataset(
            list(zip(all_paths[train_idx].tolist(), all_labels[train_idx].tolist())),
            transform=train_val_transform,
        )
        val_dataset = PathListDataset(
            list(zip(all_paths[val_idx].tolist(), all_labels[val_idx].tolist())),
            transform=train_val_transform,
        )
        test_dataset = PathListDataset(
            list(zip(all_paths[test_idx].tolist(), all_labels[test_idx].tolist())),
            transform=test_transform,
        )

    else:
        # NCT7k (Disjoint): train and test come from genuinely separate
        # patient slides (no augmentation-duplicate mechanism), so the
        # official split is kept as-is -- just carve off a 90/10 train/val
        # split from the train_val side.
        train_val_dataset = datasets.ImageFolder(train_val_path)
        test_dataset_base = datasets.ImageFolder(test_path)
        assert train_val_dataset.classes == test_dataset_base.classes, (
            f"[{dataset_name}] Class mismatch between train/val and test folders: "
            f"{train_val_dataset.classes} vs {test_dataset_base.classes}."
        )
        num_classes = len(train_val_dataset.classes)
        print(f"[{dataset_name}] Classes ({num_classes}): {train_val_dataset.classes}")

        val_size = int(len(train_val_dataset) * 0.1)
        train_size = len(train_val_dataset) - val_size
        train_dataset, val_dataset = random_split(train_val_dataset, [train_size, val_size], generator=g)
        train_dataset.dataset.transform = train_val_transform
        val_dataset.dataset.transform = train_val_transform
        test_dataset_base.transform = test_transform
        test_dataset = test_dataset_base

    train_dataset = quick_subset(train_dataset, QUICK_TEST_MAX_TRAIN)
    val_dataset = quick_subset(val_dataset, QUICK_TEST_MAX_VAL)
    test_dataset = quick_subset(test_dataset, QUICK_TEST_MAX_TEST)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, persistent_workers=True)
    return train_loader, val_loader, test_loader, num_classes


In [ ]:
def get_pretrained_teachers(num_classes):
    """ViT/DeiT/Swin, ImageNet-pretrained backbone -- NOT fine-tuned on the
    target dataset (no per-dataset training, no checkpointing needed). The
    'With KD' distillation below simply averages these three teachers' logits
    (no fine-tuning, no per-teacher weighting)."""
    teacher_vit = create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_deit = create_model('deit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_swin = create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=num_classes).cuda()
    teachers = [teacher_vit, teacher_deit, teacher_swin]
    for t in teachers:
        t.eval()
    return teachers


In [ ]:
def train_and_evaluate_lorasct(dataset_name, use_kd, teacher_models, num_classes,
                                train_loader, val_loader, test_loader):
    """Trains (or loads checkpoint for) LoRaS-CT either With KD (distilled from
    teacher_models) or Without KD (plain cross-entropy on ground-truth labels),
    then returns test Accuracy (%) only."""
    checkpoint_key = "LoRaS-CT_WithKD" if use_kd else "LoRaS-CT_WithoutKD"
    ce_criterion = nn.CrossEntropyLoss()

    set_all_seeds(SEED)
    model = HybridStudentModel(num_classes, grid_size=3).cuda()

    cached = load_checkpoint(dataset_name, checkpoint_key, model)
    if cached is None:
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
        if use_kd:
            criterion = DistillationLoss(alpha=0.5, temperature=3.0)
            model = train_model_with_distillation(model, teacher_models, train_loader, val_loader,
                                                   criterion, optimizer, num_epochs=RUN_EPOCHS)
        else:
            model = train_model_plain(model, train_loader, val_loader, ce_criterion, optimizer,
                                       num_epochs=RUN_EPOCHS)
        acc, _ = evaluate_model(model, test_loader, ce_criterion)
        print(f"  [{checkpoint_key}] Test Accuracy: {acc:.2f}%")
        save_checkpoint(dataset_name, checkpoint_key, model, extra={'Accuracy (%)': acc})
    else:
        if 'Accuracy (%)' in cached:
            acc = cached['Accuracy (%)']
        else:
            acc, _ = evaluate_model(model, test_loader, ce_criterion)

    return acc


In [ ]:
def run_dataset_pipeline(dataset_name, dataset_cfg):
    print(f"\n{'#'*80}\n# DATASET: {dataset_name}\n{'#'*80}")

    train_loader, val_loader, test_loader, num_classes = \
        load_dataset(dataset_name, dataset_cfg, seed=SEED)
    print(f"[{dataset_name}] Classes: {num_classes}")

    # Teachers: ImageNet-pretrained, NOT fine-tuned -- used only for the
    # With-KD variant, logits simply averaged (see train_model_with_distillation).
    teachers = get_pretrained_teachers(num_classes)

    print(f"\n--- LoRaS-CT | With KD ---")
    acc_with_kd = train_and_evaluate_lorasct(dataset_name, True, teachers, num_classes,
                                              train_loader, val_loader, test_loader)

    print(f"\n--- LoRaS-CT | Without KD ---")
    acc_without_kd = train_and_evaluate_lorasct(dataset_name, False, None, num_classes,
                                                 train_loader, val_loader, test_loader)

    del teachers
    torch.cuda.empty_cache()
    gc.collect()

    return {'With KD': acc_with_kd, 'Without KD': acc_without_kd}


## 9. Run all selected datasets

In [ ]:
all_results = {}

for dataset_name in DATASETS_TO_RUN:
    dataset_cfg = DATASET_CONFIGS[dataset_name]
    missing = _check_dataset_paths(dataset_cfg)
    if missing:
        print(f"[{dataset_name}] SKIPPED -- path(s) not found: {missing}")
        continue
    try:
        all_results[dataset_name] = run_dataset_pipeline(dataset_name, dataset_cfg)
    except Exception as e:
        print(f"[{dataset_name}] PIPELINE FAILED -- {type(e).__name__}: {e}")


## 10. Final results table — Accuracy (%), With KD vs Without KD

In [ ]:
results_df = pd.DataFrame(all_results).T
results_df = results_df[['With KD', 'Without KD']]
results_df['KD Gain (pp)'] = results_df['With KD'] - results_df['Without KD']
results_df = results_df.round(2)
results_df
